# Evaluating Tidally Induced Acceleration as a Novel Cause of the Lunar Crustal Asymmetry

Author: Kas Knicely (she|her)

Date of Creation: 2026 March 17

Affiliation at date of Creation: University of Alaska Fairbanks

Description: <br>
This code tests the hypotheses that tidal acceleration could have caused the lunar crustal asymmetry. The lunar crustal asymmetry is a deviation of the crustal thickness of the Moon from uniform. The near-side is ~30-40 km thick, the far-side ~50-60 km, a difference of ~20 km, and this thickness smoothly varies from a minimum on the near-side to its maximum on the far-side. This implies some ubiquitous globe-spanning phenomenon caused the thickness of the crust to change. I hypothesize that a much closer Moon may have experienced significantly greater tidal acceleration due to the Earth's bulge and rotation, resulting in a modification in the distribution of mass. <br>
We know that today the Moon is drifting away from the Earth at ~3-4 cm/yr. This is caused by tidal acceleration. The Moon induces a tidal bulge on the Earth. Because the Earth rotates, this bulge is then shifted off of the Earth/Moon axis. This shift results in a slight prograde pull on the Moon. The prograde pull moves the Moon into a higher orbit. Currently, this is an extremely small acceleration. This may have been different in the past. <br>
Early in the history of the Earth/Moon system, the Moon would have been much closer to the Earth, resulting in a larger tidal bulge on the Earth and a larger prograde acceleration. Some estimates put the Moon at ~25,000 km, compared to its present 384,400 km. The theoretical closest proximity is the Roche limit, which is ~19,000 km. That would put the Moon 20 times closer, resulting in a 400 times stronger gravitational effect (gravity varies with the square of distance). If the resulting tidal acceleration is large enough, it could result in a change in the distribution of mass within the Moon and explain the observed lunar crustal asymmetry. 

***
## 0. Import Python Libraries

This code imports the necessary libraries used for the calculations and plotting. 

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import lpmv

***
## 1. Set constants

Set universal constants. 

Note: For the bulk shear modulus max, a value of 150 GPa is selected. For the modern Earth, this gives a nearly perfect k_2 (2nd degree tidal love number; 0.295 according to Lainey [2016] (Quantification of tidal parameters from Solar System data)). It overestimates k_3; a calculated value of 0.129 compared to a measured value of 0.093 (a difference of approximately 1/3rd). 

In [45]:
g_TIA_min = 0.00911 # minimum tidally induced acceleration (TIA) necessary to produce the observed crustal asymmetry. 

# Needed for mean motion and secular acceleration calc. 
m_M = 7.3476 * 10**22 # mass of the Moon in kg.
m_E = 5.9722 * 10**24 # mass of the Earth in kg.
G   = 6.6743 * 10**(-11) # gravitational constant; units of [ m^3 / (kg s^2) ]
r_E = 6371 * 10**3 # radius of the Earth in meters. 

# Needed for tidal love number calc (plus r_E above).
mu_E_max = 150. * 10.**9. # Earth's bulk shear modulus in Pascals; this gives a perfect k_2 and a k_3 that's a little larger than it should be. 
mu_E_min = 0. # theoretical minimum bulk shear modulus; this is a completely fluid body. 
g_E = 9.81 # Earth's surface gravity in m/s^2. 
rho_E = 5515.0 # earth's bulk density in kg/m^3. 

Set the parameter space for exploration. 

In [46]:
radii = np.arange(10, 51, 1) * 1000 # set up radii in kilometers; 10,000 to 50,000 in steps of 1 km. 
lagAngle = np.arange(3, 46, 1) # set up tidal lag angles to explore; from current (3 degrees) to theoretical maximum (45). 
mu_Es = np.linspace(mu_E_min, mu_E_max, num = 50)
N = 5 # maximum degree to use in the approximation. 

Calculate love numbers. 

This is based on Taylor & Margot [2010] (Tidal evolution of close binary asteroid systems); DOI:10.1007/s10569-010-9308-0. It is identical to the k_j calculation given by Bill et al. [2005] (Improved estimate of tidal dissipation within Mars from MOLA observations of the shadow of Phobos). These give excellent agreement for k_2 for the modern Earth and slightly overestimates k_3 for the modern Earth. 

In [50]:
def loveNumberCalc(j, mu_p, g_p, rho_p, R_p): 
    # Calculate the tidal love number for the Earth/Moon system. 
    # This is based on Taylor & Margot [2010] (Tidal evolution of close binary asteroid systems); DOI:10.1007/s10569-010-9308-0. 
    # It is identical to the k_j given by Bill et al. [2005] (Improved estimate of tidal dissipation within Mars from MOLA observations of the shadow of Phobos). 

    # j - degree of the love number. 
    # mu_p - rigidity or shear modulus. 
    # g_p - surface gravity of the primary (Earth).
    # rho_p - average density of the primary (Earth).
    # R_p - average radius of the primary (Earth). 

    # lN - love number output. 
    
    part1 = 3. / ( 2.*(j-1) )

    part2b = ( 2.*j**2. + 4.*j + 3. ) * mu_p / ( j*g_p*rho_p*R_p )

    part2a = 1. / ( 1. + part2b )
    
    lN = part1 * part2a
    
    return lN

Calculate the derivatives of the legendre polynomials. 

This initial version is based on the already worked out derivatives from Pou et al. [2022] (Tidal constraints on the Martian Interior). Long-term, I plan to update this to a generalized legendre polynomial derivative calculator. Some early attempts have been made, but consistent results elude me. 

In [5]:
def legendrePolynomialDerivative(gamma, j):
    # Function to get the derivative of the legendre polynomial. 
    # This was taken from Pou et al. [2022] (Tidal constraints on the Martian Interior). 
    # Ultimately, this should get replaced with code that can get any lPD. 

    # Convert gamma from degrees to radians. 
    rad = np.deg2rad(gamma)
    
    if j == 2:
        lPD = -(3./2.) * np.sin(2.*rad)
    elif j == 3:
        lPD = -(3./8.) * ( np.sin(rad) + 5.*np.sin(3.*rad) )
    elif j == 4:
        lPD = -(5./16.) * ( 2.*np.sin(2.*rad) + 7.*np.sin(4.*rad) )
    elif j == 5:
        lPD = -(15./128.) * ( 2.*np.sin(rad) + 7.*np.sin(3.*rad) + 21.*np.sin(5.*rad) )
    elif j == 6:
        # Acquired from 10.1007/s10569-010-9308-0; Taylor & Margot [2010] (Tidal evolution of close binary asteroid systems)
        lPD = -(21./256.) * ( 5.*np.sin(2.*rad) + 12.*np.sin(4.*rad) + 33.*np.sin(6.*rad) )

    # # Get the legendre polynomial. 
    # x_value = np.cos(gamma)
    # lP = lpmv(0, j, x_value)

    # # Take the derivative of the legendre polynomial. 
    # lPD 

    return lPD

In [9]:
### NEEDS TESTING ###
from scipy.special import legendre
def legendrePolynomialDerivative_v2(gamma, j):
    # Function to get the derivative of the legendre polynomial. 
    # This was taken from Pou et al. [2022] (Tidal constraints on the Martian Interior). 
    # Ultimately, this should get replaced with code that can get any lPD. 
    # https://stackoverflow.com/questions/42153202/getting-the-derivatives-of-legendre-polynomials-in-python

    # Get the legendre polynomial. 
    poly = legendre(j)

    # Take the derivative of the legendre polynomial. 
    polyd = poly.deriv()
    # print(f"polys = {polyd}")
    
    # Evaluate the derivative of the legendre polynomial. 
    rad = np.deg2rad(gamma) # convert from degrees to radians. 
    x_value = np.sin(rad)
    evald = np.polyval(polyd, x_value)

    return evald

In [11]:
### NEEDS TESTING ###
from scipy.special import legendre_p
def legendrePolynomialDerivative_v3(gamma, j):
    # Function to get the derivative of the legendre polynomial. 
    # This was taken from Pou et al. [2022] (Tidal constraints on the Martian Interior). 
    # Ultimately, this should get replaced with code that can get any lPD. 
    # https://stackoverflow.com/questions/42153202/getting-the-derivatives-of-legendre-polynomials-in-python

    # j - degre
    # gamma - angle

    
    # lPD = legendre_p(j, 

    # Get the legendre polynomial. 
    
    poly = legendre(j)

    # Take the derivative of the legendre polynomial. 
    polyd = poly.deriv()
    # print(f"polys = {polyd}")
    
    # Evaluate the derivative of the legendre polynomial. 
    rad = np.deg2rad(gamma) # convert from degrees to radians. 
    x_value = np.cos(rad)
    evald = np.polyval(polyd, x_value)

    return evald

In [12]:
lPD = legendrePolynomialDerivative(3, 2)
lPD_v2 = legendrePolynomialDerivative_v2(3,2)
lPD_v3 = legendrePolynomialDerivative_v3(3,2)
print(lPD, lPD_v2, lPD_v3)
lPD = legendrePolynomialDerivative(2, 2)
lPD_v2 = legendrePolynomialDerivative_v2(2,2)
lPD_v3 = legendrePolynomialDerivative_v3(2,2)
print(lPD, lPD_v2, lPD_v3)
lPD = legendrePolynomialDerivative(30, 2)
lPD_v2 = legendrePolynomialDerivative_v2(30,2)
lPD_v3 = legendrePolynomialDerivative_v3(30,2)
print(lPD, lPD_v2, lPD_v3)
lPD = legendrePolynomialDerivative(30, 3)
lPD_v2 = legendrePolynomialDerivative_v2(30,3)
lPD_v3 = legendrePolynomialDerivative_v3(30,3)
print(lPD, lPD_v2, lPD_v3)

-0.1567926949014802 0.1570078687288315 2.9958886042637216
-0.10463471061618795 0.1046984901075029 2.998172481057287
-1.299038105676658 1.4999999999999998 2.598076211353316
-2.0625 0.37499999999999933 4.125000000000001


***
# 2.0 Calculate the Tidally Induced Acceleration

Define several important functions. 

In [61]:
# Needs testing!!!
def meanMotionCalc(G, m_p, m_s, r):
    # Function to calculate the mean motion. 
    # n = sqrt( G(m_p+m_s) / r^3 )
    # n - mean motion; units of s^-1
    # G - gravitational constant; m^3 / (kg s^2)
    # m_p - mass of the primary (kg)
    # m_s - mass of the secondary (kg)
    # r - orbital radius (m)

    n = np.sqrt( G*(m_p+m_s) / r**3 )
    
    return n

In [62]:
n = meanMotionCalc(G, m_E, m_M, radii)
n

array([20.08747255, 17.41150326, 15.28105811, 13.55221453, 12.12643778,
       10.9342351 ,  9.92533839,  9.06258127,  8.31795446,  7.66999171,
        7.10199403,  6.60079666,  6.15589601,  5.7588205 ,  5.40266991,
        5.08177325,  4.7914314 ,  4.52772092,  4.28734319,  4.06750691,
        3.86583589,  3.68029605,  3.50913704,  3.35084536,  3.20410634,
        3.06777306,  2.940841  ,  2.82242695,  2.71175158,  2.6081249 ,
        2.51093407,  2.41963307,  2.33373404,  2.25279987,  2.17643791,
        2.10429453,  2.03605051,  1.97141705,  1.91013226,  1.85195818,
        1.79667816])

In [63]:
### NEEDS TESTING ###
def secularAccelerationCalc(m_p, m_s, n, N, gamma, A, r, mu, g, rho):
    # Function to calculate the secular acceleration. 
    # s = -3/2 n^2 alpha [sum from j=2 to N of ( k_j * F_j(gamma) * (A/r) ^ (2j+1) ) ]
    # s - secular acceleration; value output. units of 1 / s^2. 
    # n - mean motion; this can be a scalar or a vector. ; units of s^-1
    # alpha - constant given by the mass of the secondary divided by the mass of the primary. 
    # N - highest degree to calculate. 
    # k_j - tidal love numbers. 
    # F_j - legrendre polynomial derivative. 
    # gamma - tidal lag angle in degrees. 
    # A - radius of Earth in meters. 
    # r - orbital radius in meters; this should be a scalar. 
    # mu - bulk shear modulus of primary (Earth). 
    # rho - bulk density of primary (Earth). 

    # Check that the radius is given as a scalar and not a vector. 
    if not np.isscalar(r):
        print("WARNING!!!\nWARNING!!!\nWARNING!!!\n")
        print("Given radius is not a scalar. Please give the radius as a scalar.")

        return
        
    # Calculate alpha. 
    alpha = m_s / m_p

    # Calculate the summation term. 
    # summation = sum from j=2 to N of ( k_j * F_j(gamma) * (A/r)^(2j+1) )
    summation = 0.0
    for j in range(2,N+1):
        k_j = loveNumberCalc(j, mu, g, rho, A)
        F_j = legendrePolynomialDerivative(gamma, j)
        print(f"k_{j} = {k_j}")
        temp = k_j * F_j * (A/r)**(2*j+1)
        summation = summation + temp
    
    # Make the final calculation. 
    s = (-3./2.) * n**2.0 * alpha * summation
    
    return s

In [64]:
s = secularAccelerationCalc(m_E, m_M, n[0], N, lagAngle[0], r_E, radii[1], mu_E_max, g_E, rho_E)
s

k_2 = 0.2921577883030757
k_3 = 0.12960122749321742
k_4 = 0.07635280859787337
k_5 = 0.05099521614623986


np.float64(7.263808666885578e+29)

In [20]:
radii

array([10000, 11000, 12000, 13000, 14000, 15000, 16000, 17000, 18000,
       19000, 20000, 21000, 22000, 23000, 24000, 25000, 26000, 27000,
       28000, 29000, 30000, 31000, 32000, 33000, 34000, 35000, 36000,
       37000, 38000, 39000, 40000, 41000, 42000, 43000, 44000, 45000,
       46000, 47000, 48000, 49000, 50000])

In [ ]:
### NEEDS TESTING ###
def g_TIA_Calc():
    # Function to calculate the tidally induced acceleration. 
    # s - secular acceleration. 
    # r - orbital radius. 

    s = secularAccelerationCalc()

    g_TIA = s * r
    
    return g_TIA